# 🐼 RAG Pipeline — Kung Fu Panda Edition

Complete walkthrough of a Retrieval-Augmented Generation pipeline using Kung Fu Panda characters and plot as the document corpus. Includes 8 documents (6 KFP + 2 noisy real-world kung fu docs) to demonstrate noise resilience.

```
📄 Raw Documents  →  Stage 1: Extraction  →  Stage 2: Chunking
→  Stage 3: Indexing (BM25 + Dense + Hybrid)  ← OFFLINE
→  Stage 4: Retrieval  →  Stage 5: Reranking  →  Stage 6: Generation  ← ONLINE
```

⚡ **Runtime:** `Runtime → Change runtime type → T4 GPU`

## Install dependencies

In [1]:
!pip install -q rank_bm25 sentence-transformers transformers accelerate
!pip install -q numpy scikit-learn
print('✅ All packages installed')


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
✅ All packages installed


## Imports and helpers

In [2]:
import numpy as np
import textwrap
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from sklearn.metrics.pairwise import cosine_similarity

def banner(title, char='═', width=70):
    """Print a section header banner."""
    print(f'\n{char*width}\n  {title}\n{char*width}')

def section(title):
    """Print a subsection header."""
    print(f'\n  ┌─ {title} {"-"*(60-len(title))}')

print('✅ Imports ready')

/Users/andrevarilla/Git Repos/natural-language-processsing/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Imports ready


## Stage 1: Raw documents  8 documents across three categories:   • KFP characters  (doc_A – doc_D) : plot-relevant, query targets   • KFP story arcs  (doc_E – doc_F) : plot-relevant, query targets   • Noisy / general (doc_G – doc_H) : real-world kung fu, not KFP     → these test whether retrieval can resist irrelevant documents

In [3]:
banner('STAGE 1 — TEXT EXTRACTION (simulated)')

raw_documents = {

    # ── KFP character docs ──────────────────────────────────────────────

    'doc_A_po': """
    Po is a giant panda who works as a noodle waiter in his adoptive father Mr Ping's
    noodle shop in the Valley of Peace. Po is an enthusiastic fan of kung fu and of
    the Furious Five. He is clumsy, overweight, and has no formal martial arts training.
    Master Oogway chooses Po as the Dragon Warrior despite the protests of Master Shifu
    and the Furious Five. Po's greatest strength is his resilience and his unorthodox
    approach to kung fu. He ultimately defeats Tai Lung by using the Wuxi Finger Hold.
    Po learns that the Dragon Scroll contains no secret — the power was always within himself.
    In Kung Fu Panda 2, Po discovers he was adopted and learns his biological parents
    sacrificed themselves to save him from Lord Shen's massacre of the pandas.
    """,

    'doc_B_tigress': """
    Tigress is the unofficial leader of the Furious Five and the strongest fighter
    among them. She trained under Master Shifu at the Jade Palace since childhood.
    Tigress harboured a deep belief that she, not Po, should have been chosen as the
    Dragon Warrior. Her fighting style is direct and powerful, relying on strength,
    speed, and iron-hard discipline. Tigress was raised in an orphanage before Shifu
    adopted her; her early life was marked by loneliness and the fear that no one
    could ever love a tiger. She is deeply loyal to Shifu and to the Valley of Peace.
    Over time Tigress develops respect for Po, though she rarely expresses it openly.
    Tigress's signature move is a powerful direct strike that can shatter boulders.
    """,

    'doc_C_mantis': """
    Mantis is the smallest member of the Furious Five but arguably the fastest.
    His fighting style is based on praying mantis kung fu, emphasising speed,
    precision strikes, and the ability to apply pressure to nerve points to
    temporarily paralyse opponents. Mantis is often the comic relief of the group,
    cracking jokes and complaining about his size. Despite his small stature,
    Mantis is strong enough to hold back a rope attached to a fully loaded boat.
    He is also skilled at acupuncture, which he uses to help Po manage his injuries.
    Mantis has a history of bad luck with relationships and is easily irritated
    when others underestimate him because of his size.
    """,

    'doc_D_tai_lung': """
    Tai Lung is the primary antagonist of the first Kung Fu Panda film. He was
    once Master Shifu's most prized student and trained his entire life to receive
    the Dragon Scroll. When Master Oogway refused to give Tai Lung the scroll,
    declaring he lacked inner peace, Tai Lung flew into a rage and attacked the
    Valley of Peace. He was defeated and imprisoned in Chorh-Gom Prison for twenty
    years. Tai Lung's fighting style combines Snow Leopard kung fu with nerve
    attack techniques that can temporarily disable an opponent's limbs. His deepest
    wound is not physical but emotional: he craved Shifu's approval and never
    received it. Tai Lung is ultimately defeated by Po's Wuxi Finger Hold.
    """,

    # ── KFP story arc docs ───────────────────────────────────────────────

    'doc_E_furious_five': """
    The Furious Five are the five greatest kung fu warriors of the Valley of Peace:
    Tigress, Crane, Mantis, Viper, and Monkey. They were trained by Master Shifu
    at the Jade Palace. Each member has a distinct fighting style drawn from their
    animal form: Tigress uses direct power, Crane relies on evasion and aerial moves,
    Mantis uses speed and nerve strikes, Viper uses flexibility and rapid strikes,
    and Monkey blends acrobatics with unpredictable attacks. Initially the Furious
    Five resent Po's appointment as Dragon Warrior. Over the course of the film they
    come to respect his courage and creativity. The Five are defeated by Tai Lung
    one by one when they attempt to stop him before he reaches the Valley of Peace.
    """,

    'doc_F_shifu_oogway': """
    Master Shifu is a red panda and the headmaster of the Jade Palace. He trained
    Tai Lung, and later the Furious Five. Shifu struggles with guilt over Tai Lung's
    fall, feeling responsible for creating the monster he must now stop. Oogway
    forces Shifu to train Po, which Shifu initially does through punishing exercises
    designed to make Po quit. Shifu eventually discovers that Po can be motivated
    through food, and uses this insight to teach him kung fu rapidly. Master Oogway
    is an ancient tortoise revered as the greatest kung fu master who ever lived.
    Oogway chose Po as the Dragon Warrior based on a vision and his belief that
    there are no accidents. Oogway passes away peacefully, ascending as peach
    blossom petals, shortly after naming Po the Dragon Warrior.
    """,

    # ── Noisy docs: real-world kung fu, irrelevant to KFP plot ──────────
    # These are designed to score well on generic 'kung fu' queries but
    # poorly on character-specific or KFP-plot queries.

    'doc_G_shaolin_history': """
    Shaolin kung fu originated at the Shaolin Monastery in Henan Province, China,
    around the 5th century CE. According to tradition, the Indian monk Bodhidharma
    introduced a series of exercises to strengthen the monks' bodies for meditation.
    Over centuries these exercises evolved into a comprehensive martial art system.
    Shaolin kung fu encompasses over a hundred distinct styles, divided into
    northern and southern schools. Northern styles emphasise high kicks and long
    stances; southern styles favour low stances, short punches, and close-range
    combat. The monastery was twice destroyed and rebuilt, with its martial
    traditions surviving through travelling monks and lay disciples. Modern Shaolin
    kung fu is practised worldwide both as a martial art and as a performance art.
    The five animal styles — tiger, crane, leopard, snake, and dragon — are among
    the most famous forms associated with Shaolin training.
    """,

    'doc_H_wingchun_techniques': """
    Wing Chun is a southern Chinese kung fu style emphasising economy of motion,
    simultaneous attack and defence, and close-range combat. It was supposedly
    developed by a Buddhist nun named Ng Mui and taught to a young woman named
    Yim Wing Chun, from whom the style takes its name. Wing Chun uses a narrow
    stance with knees turned inward for structural integrity. The chain punch,
    a rapid alternating series of punches aimed at the centreline, is its signature
    technique. Wing Chun practitioners use a training dummy called the Mook Yan Jong
    to develop hardness and precision. The style avoids brute force and instead
    redirects incoming attacks using contact reflexes developed through chi sao,
    or sticky hands, drills. Bruce Lee studied Wing Chun under Ip Man before
    developing his own Jeet Kune Do philosophy.
    """,
}

# ── Display summary ──────────────────────────────────────────────────────────
for doc_id, text in raw_documents.items():
    clean = ' '.join(text.split())
    tag = '🐼 KFP' if 'noise' not in doc_id and doc_id not in ['doc_G_shaolin_history','doc_H_wingchun_techniques'] else '🔴 NOISY'
    tag = '🔴 NOISY' if doc_id in ['doc_G_shaolin_history', 'doc_H_wingchun_techniques'] else '🐼 KFP'
    print(f'  {tag}  [{doc_id}]: {len(clean.split())} words')
    print(f'         Preview: "{clean[:80]}..."')
    print()


══════════════════════════════════════════════════════════════════════
  STAGE 1 — TEXT EXTRACTION (simulated)
══════════════════════════════════════════════════════════════════════
  🐼 KFP  [doc_A_po]: 133 words
         Preview: "Po is a giant panda who works as a noodle waiter in his adoptive father Mr Ping'..."

  🐼 KFP  [doc_B_tigress]: 122 words
         Preview: "Tigress is the unofficial leader of the Furious Five and the strongest fighter a..."

  🐼 KFP  [doc_C_mantis]: 111 words
         Preview: "Mantis is the smallest member of the Furious Five but arguably the fastest. His ..."

  🐼 KFP  [doc_D_tai_lung]: 115 words
         Preview: "Tai Lung is the primary antagonist of the first Kung Fu Panda film. He was once ..."

  🐼 KFP  [doc_E_furious_five]: 120 words
         Preview: "The Furious Five are the five greatest kung fu warriors of the Valley of Peace: ..."

  🐼 KFP  [doc_F_shifu_oogway]: 129 words
         Preview: "Master Shifu is a red panda and the headmaster of th

## Stage 1 display: word count bar (aesthetic, separate from logic)

In [4]:
print('\n  Word count per document:')
print('  ' + '─'*50)
for doc_id, text in raw_documents.items():
    wc = len(text.split())
    bar = '█' * (wc // 5)
    label = '(noisy)' if doc_id in ['doc_G_shaolin_history','doc_H_wingchun_techniques'] else ''
    print(f'  {doc_id:<28} {bar:<20} {wc} {label}')


  Word count per document:
  ──────────────────────────────────────────────────
  doc_A_po                     ██████████████████████████ 133 
  doc_B_tigress                ████████████████████████ 122 
  doc_C_mantis                 ██████████████████████ 111 
  doc_D_tai_lung               ███████████████████████ 115 
  doc_E_furious_five           ████████████████████████ 120 
  doc_F_shifu_oogway           █████████████████████████ 129 
  doc_G_shaolin_history        ███████████████████████████ 137 (noisy)
  doc_H_wingchun_techniques    ██████████████████████████ 132 (noisy)


## Stage 2: Chunking logic

In [6]:
def chunk_text(text, chunk_size=80, overlap=0):
    """
    Split text into overlapping word-level chunks.

    Parameters
    ----------
    text       : str   raw text to split
    chunk_size : int   number of words per chunk
    overlap    : int   number of words repeated from the previous chunk
                       prevents answers at boundaries from being split

    Returns
    -------
    list[str]  — list of chunk strings, each at least 10 words long
    """
    words = text.split()
    chunks = []
    step = chunk_size - overlap          # advance this many words per step
    for start in range(0, len(words), step):
        chunk_words = words[start : start + chunk_size]
        if len(chunk_words) < 10:        # skip tiny trailing fragments
            break
        chunks.append(' '.join(chunk_words))
    return chunks


banner('STAGE 2 — CHUNKING  (overlap tradeoff demo)')

# ── Raw document word counts ─────────────────────────────────────────────────
print('\n━━ RAW DOCUMENTS ━━')
for doc_id, text in raw_documents.items():
    words = text.split()
    print(f'\n  [{doc_id}]  ({len(words)} words)')
    print(f'    {" ".join(words[:15])} ...')

# ── No overlap ───────────────────────────────────────────────────────────────
print('\n━━ CHUNKS  (overlap=0) ━━')
for doc_id, text in raw_documents.items():
    chunks = chunk_text(text, chunk_size=80, overlap=0)
    print(f'\n  [{doc_id}]  →  {len(chunks)} chunks')
    for i, c in enumerate(chunks):
        words = c.split()
        print(f'    chunk {i}: [{words[0]} ... {words[-1]}]  ({len(words)} words)')

# ── With overlap ─────────────────────────────────────────────────────────────
print('\n━━ CHUNKS  (overlap=20) ━━')
for doc_id, text in raw_documents.items():
    chunks = chunk_text(text, chunk_size=80, overlap=20)
    print(f'\n  [{doc_id}]  →  {len(chunks)} chunks')
    for i, c in enumerate(chunks):
        words = c.split()
        print(f'    chunk {i}: [{words[0]} ... {words[-1]}]  ({len(words)} words)')


══════════════════════════════════════════════════════════════════════
  STAGE 2 — CHUNKING  (overlap tradeoff demo)
══════════════════════════════════════════════════════════════════════

━━ RAW DOCUMENTS ━━

  [doc_A_po]  (133 words)
    Po is a giant panda who works as a noodle waiter in his adoptive father ...

  [doc_B_tigress]  (122 words)
    Tigress is the unofficial leader of the Furious Five and the strongest fighter among them. ...

  [doc_C_mantis]  (111 words)
    Mantis is the smallest member of the Furious Five but arguably the fastest. His fighting ...

  [doc_D_tai_lung]  (115 words)
    Tai Lung is the primary antagonist of the first Kung Fu Panda film. He was ...

  [doc_E_furious_five]  (120 words)
    The Furious Five are the five greatest kung fu warriors of the Valley of Peace: ...

  [doc_F_shifu_oogway]  (129 words)
    Master Shifu is a red panda and the headmaster of the Jade Palace. He trained ...

  [doc_G_shaolin_history]  (137 words)
    Shaolin kung fu 

## Stage 2 display: boundary problem visualised (aesthetic)

In [7]:
banner('CHUNKING TRADEOFF — Boundary Problem (doc_B_tigress)')

demo_text  = ' '.join(raw_documents['doc_B_tigress'].split())
CHUNK_SIZE = 30

chunks_no_overlap   = chunk_text(demo_text, chunk_size=CHUNK_SIZE, overlap=0)
chunks_with_overlap = chunk_text(demo_text, chunk_size=CHUNK_SIZE, overlap=10)

print('\n  30-word chunks, NO overlap:')
for i, c in enumerate(chunks_no_overlap):
    words = c.split()
    boundary = words[-3:]
    print(f'    Chunk {i}: "{" ".join(words[:-3])} [{" ".join(boundary)}]"')

print('\n  30-word chunks, 10-word OVERLAP:')
for i, c in enumerate(chunks_with_overlap):
    words = c.split()
    if i == 0:
        print(f'    Chunk {i}: "{" ".join(words)}"')
    else:
        repeated  = words[:10]
        new_words = words[10:]
        print(f'    Chunk {i}: "[OVERLAP: {" ".join(repeated[:4])}...] + {" ".join(new_words[:6])}..."')

n0  = len(chunks_no_overlap)
n10 = len(chunks_with_overlap)
print(f'\n  ⚠️  No overlap: {n0} chunks   |   10-word overlap: {n10} chunks  (+{n10-n0} extra)')
print('     Overlap repeats tail of each chunk as head of next → boundary answers always retrievable.')


══════════════════════════════════════════════════════════════════════
  CHUNKING TRADEOFF — Boundary Problem (doc_B_tigress)
══════════════════════════════════════════════════════════════════════

  30-word chunks, NO overlap:
    Chunk 0: "Tigress is the unofficial leader of the Furious Five and the strongest fighter among them. She trained under Master Shifu at the Jade Palace since childhood. Tigress [harboured a deep]"
    Chunk 1: "belief that she, not Po, should have been chosen as the Dragon Warrior. Her fighting style is direct and powerful, relying on strength, speed, and iron-hard discipline. [Tigress was raised]"
    Chunk 2: "in an orphanage before Shifu adopted her; her early life was marked by loneliness and the fear that no one could ever love a tiger. She is [deeply loyal to]"
    Chunk 3: "Shifu and to the Valley of Peace. Over time Tigress develops respect for Po, though she rarely expresses it openly. Tigress's signature move is a powerful direct [strike that can]"

## Build final chunk corpus for indexing

In [8]:
# chunk_size=60, overlap=15 is the working corpus for all downstream stages
ALL_CHUNKS = []
for doc_id, text in raw_documents.items():
    clean      = ' '.join(text.split())
    doc_chunks = chunk_text(clean, chunk_size=60, overlap=15)
    for i, c in enumerate(doc_chunks):
        ALL_CHUNKS.append({
            'id'    : f'{doc_id}_c{i}',
            'text'  : c,
            'source': doc_id,
        })

print(f'\n  ✅ Final corpus: {len(ALL_CHUNKS)} chunks ready for indexing')
print(f'     (chunk_size=60, overlap=15)')
print()
for c in ALL_CHUNKS[:6]:
    print(f'  [{c["id"]}]: "{c["text"][:90]}..."')


  ✅ Final corpus: 24 chunks ready for indexing
     (chunk_size=60, overlap=15)

  [doc_A_po_c0]: "Po is a giant panda who works as a noodle waiter in his adoptive father Mr Ping's noodle s..."
  [doc_A_po_c1]: "martial arts training. Master Oogway chooses Po as the Dragon Warrior despite the protests..."
  [doc_A_po_c2]: "Po learns that the Dragon Scroll contains no secret — the power was always within himself...."
  [doc_B_tigress_c0]: "Tigress is the unofficial leader of the Furious Five and the strongest fighter among them...."
  [doc_B_tigress_c1]: "style is direct and powerful, relying on strength, speed, and iron-hard discipline. Tigres..."
  [doc_B_tigress_c2]: "Shifu and to the Valley of Peace. Over time Tigress develops respect for Po, though she ra..."


## Stage 3a: BM25 sparse index (OFFLINE)

In [9]:
banner('STAGE 3a — SPARSE INDEX (BM25)  ← OFFLINE')
print()
print('  BM25 tokenises every chunk into words.')
print('  Internally it stores:')
print('    • idf dict      → how rare each term is across all chunks')
print('    • doc_freqs     → how often each term appears in each chunk')
print('  No model, no GPU, no training needed.')
print()

# Tokenise: lowercase + whitespace split must match query tokenisation later
tokenised_chunks = [c['text'].lower().split() for c in ALL_CHUNKS]

# BM25Okapi.fit() builds the full index in one pass over the tokenised corpus
bm25_index = BM25Okapi(tokenised_chunks)

# ── Show posting lists (which chunks contain each term) ──────────────────────
section('Posting lists — which chunks contain each query term')
sample_words = ['tigress', 'tai lung', 'dragon', 'mantis', 'shifu', 'shaolin', 'wing chun']
for word in sample_words:
    containing = [ALL_CHUNKS[i]['id'] for i, toks in enumerate(tokenised_chunks)
                  if word in ' '.join(toks)]
    print(f'  "{word:<12}" → {len(containing)} chunks: {containing[:4]}')

print()
print(f'  ✅ BM25 index built: {bm25_index.corpus_size} chunks, '
      f'vocab={len(bm25_index.idf)} unique tokens')


══════════════════════════════════════════════════════════════════════
  STAGE 3a — SPARSE INDEX (BM25)  ← OFFLINE
══════════════════════════════════════════════════════════════════════

  BM25 tokenises every chunk into words.
  Internally it stores:
    • idf dict      → how rare each term is across all chunks
    • doc_freqs     → how often each term appears in each chunk
  No model, no GPU, no training needed.


  ┌─ Posting lists — which chunks contain each query term --------
  "tigress     " → 4 chunks: ['doc_B_tigress_c0', 'doc_B_tigress_c1', 'doc_B_tigress_c2', 'doc_E_furious_five_c0']
  "tai lung    " → 7 chunks: ['doc_A_po_c1', 'doc_D_tai_lung_c0', 'doc_D_tai_lung_c1', 'doc_D_tai_lung_c2']
  "dragon      " → 9 chunks: ['doc_A_po_c0', 'doc_A_po_c1', 'doc_A_po_c2', 'doc_B_tigress_c0']
  "mantis      " → 5 chunks: ['doc_C_mantis_c0', 'doc_C_mantis_c1', 'doc_C_mantis_c2', 'doc_E_furious_five_c0']
  "shifu       " → 10 chunks: ['doc_A_po_c1', 'doc_B_tigress_c0', 'doc_B_tigress_c

## BM25 index inspection (attributes)

In [12]:
banner('BM25 INDEX — OBJECT INSPECTION')

print(f'  Type          : {type(bm25_index).__name__}')
print(f'  corpus_size   : {bm25_index.corpus_size}  ← total chunks indexed')
print(f'  avgdl         : {bm25_index.avgdl:.2f}     ← avg chunk length in words')
print(f'  doc_len       : {bm25_index.doc_len}')

print(f'\n  ── Hyperparameters ──')
print(f'  k1      : {bm25_index.k1}    ← TF saturation (higher = raw counts matter more)')
print(f'  b       : {bm25_index.b}   ← length penalty (1=full, 0=none)')
print(f'  epsilon : {bm25_index.epsilon}  ← IDF floor for terms appearing in every chunk')

print(f'\n  ── IDF ({len(bm25_index.idf)} terms) ──')
top_idf = sorted(bm25_index.idf.items(), key=lambda x: x[1], reverse=True)
print('  Top 10 most discriminative:')
for term, score in top_idf[:10]:
    print(f'    {term:<28} IDF={score:.4f}')
print('  Bottom 5 (near epsilon floor, appear in almost all chunks):')
for term, score in top_idf[-5:]:
    print(f'    {term:<28} IDF={score:.4f}')

print(f'\n  ── doc_freqs (top terms per chunk, first 6 chunks) ──')
for i, freq_dict in enumerate(bm25_index.doc_freqs[:6]):
    top_terms = sorted(freq_dict.items(), key=lambda x: x[1], reverse=True)[:5]
    print(f'  chunk {i} ({bm25_index.doc_len[i]} words): {dict(top_terms)}')


══════════════════════════════════════════════════════════════════════
  BM25 INDEX — OBJECT INSPECTION
══════════════════════════════════════════════════════════════════════
  Type          : BM25Okapi
  corpus_size   : 24  ← total chunks indexed
  avgdl         : 51.62     ← avg chunk length in words
  doc_len       : [60, 60, 43, 60, 60, 32, 60, 60, 21, 60, 60, 25, 60, 60, 30, 60, 60, 39, 60, 60, 47, 60, 60, 42]

  ── Hyperparameters ──
  k1      : 1.5    ← TF saturation (higher = raw counts matter more)
  b       : 0.75   ← length penalty (1=full, 0=none)
  epsilon : 0.25  ← IDF floor for terms appearing in every chunk

  ── IDF (500 terms) ──
  Top 10 most discriminative:
    giant                        IDF=2.7515
    works                        IDF=2.7515
    noodle                       IDF=2.7515
    waiter                       IDF=2.7515
    adoptive                     IDF=2.7515
    father                       IDF=2.7515
    mr                           IDF=2.7515
    p

## BM25 full structure as JSON + save + load

In [11]:
import json, pickle

banner('BM25 INDEX — STRUCTURE → SAVE → LOAD')

# ── 1. Print full structure ──────────────────────────────────────────────────
bm25_snapshot = {
    'corpus_size' : bm25_index.corpus_size,
    'avgdl'       : round(bm25_index.avgdl, 4),
    'average_idf' : round(bm25_index.average_idf, 4),
    'k1'          : bm25_index.k1,
    'b'           : bm25_index.b,
    'epsilon'     : bm25_index.epsilon,
    'doc_len'     : bm25_index.doc_len,
    'vocab_size'  : len(bm25_index.idf),
    # All term IDF scores, sorted by score descending
    'idf'         : {k: round(v, 6) for k, v in
                     sorted(bm25_index.idf.items(), key=lambda x: x[1], reverse=True)},
    # Per-chunk term frequencies (this IS the forward index)
    'doc_freqs'   : [dict(sorted(d.items(), key=lambda x: x[1], reverse=True))
                     for d in bm25_index.doc_freqs],
}
print('\n━━ STRUCTURE ━━')
print(json.dumps(bm25_snapshot, indent=2))

# ── 2. Save (pickle serialises the full BM25Okapi object) ───────────────────
print('\n━━ SAVE ━━')
with open('bm25_index.pkl', 'wb') as f:
    pickle.dump(bm25_index, f)
print('  saved → bm25_index.pkl  (full BM25Okapi object, all attributes)')

# ── 3. Load and verify ───────────────────────────────────────────────────────
print('\n━━ LOAD ━━')
with open('bm25_index.pkl', 'rb') as f:
    bm25_loaded = pickle.load(f)
print(f'  corpus_size : {bm25_loaded.corpus_size}')
print(f'  vocab size  : {len(bm25_loaded.idf)}')
print(f'  avgdl       : {bm25_loaded.avgdl}')
assert bm25_loaded.corpus_size == bm25_index.corpus_size
assert bm25_loaded.idf         == bm25_index.idf
print('  ✅ loaded BM25 index matches original')


══════════════════════════════════════════════════════════════════════
  BM25 INDEX — STRUCTURE → SAVE → LOAD
══════════════════════════════════════════════════════════════════════

━━ STRUCTURE ━━
{
  "corpus_size": 24,
  "avgdl": 51.625,
  "average_idf": 2.3355,
  "k1": 1.5,
  "b": 0.75,
  "epsilon": 0.25,
  "doc_len": [
    60,
    60,
    43,
    60,
    60,
    32,
    60,
    60,
    21,
    60,
    60,
    25,
    60,
    60,
    30,
    60,
    60,
    39,
    60,
    60,
    47,
    60,
    60,
    42
  ],
  "vocab_size": 500,
  "idf": {
    "giant": 2.751535,
    "works": 2.751535,
    "noodle": 2.751535,
    "waiter": 2.751535,
    "adoptive": 2.751535,
    "father": 2.751535,
    "mr": 2.751535,
    "ping's": 2.751535,
    "shop": 2.751535,
    "enthusiastic": 2.751535,
    "fan": 2.751535,
    "clumsy,": 2.751535,
    "overweight,": 2.751535,
    "formal": 2.751535,
    "strength": 2.751535,
    "resilience": 2.751535,
    "unorthodox": 2.751535,
    "approach": 2.751535,

## Stage 3b: Dense embedding index (OFFLINE)

In [13]:
banner('STAGE 3b — DENSE INDEX (Sentence Embeddings)  ← OFFLINE')
print()
print('  A pre-trained encoder maps each chunk → a fixed-size vector.')
print('  Semantically similar chunks cluster nearby in this vector space.')
print('  Stored as a 2D float32 matrix: shape = (n_chunks, embedding_dim)')
print()

print('  Loading encoder: all-MiniLM-L6-v2  (80 MB, 384-dim output)...')
encoder     = SentenceTransformer('all-MiniLM-L6-v2')
chunk_texts = [c['text'] for c in ALL_CHUNKS]
print(f'  Encoding {len(chunk_texts)} chunks...')
dense_index = encoder.encode(chunk_texts, show_progress_bar=True, batch_size=16)

print(f'\n  Matrix shape : {dense_index.shape}  = {dense_index.shape[0]} chunks × {dense_index.shape[1]} dims')
print(f'  dtype        : {dense_index.dtype}')
print(f'  RAM          : {dense_index.nbytes / 1024:.1f} KB')
print(f'\n  First chunk embedding (first 8 dims): {dense_index[0][:8].round(4)}')
print(f'  ✅ Dense index built.')


══════════════════════════════════════════════════════════════════════
  STAGE 3b — DENSE INDEX (Sentence Embeddings)  ← OFFLINE
══════════════════════════════════════════════════════════════════════

  A pre-trained encoder maps each chunk → a fixed-size vector.
  Semantically similar chunks cluster nearby in this vector space.
  Stored as a 2D float32 matrix: shape = (n_chunks, embedding_dim)

  Loading encoder: all-MiniLM-L6-v2  (80 MB, 384-dim output)...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6421.13it/s]


  Encoding 24 chunks...


Batches: 100%|██████████| 2/2 [00:04<00:00,  2.37s/it]


  Matrix shape : (24, 384)  = 24 chunks × 384 dims
  dtype        : float32
  RAM          : 36.0 KB

  First chunk embedding (first 8 dims): [-0.0243  0.005   0.0103  0.0034 -0.0276 -0.0496  0.091  -0.0963]
  ✅ Dense index built.


## Dense index full structure + save + load

In [14]:
banner('DENSE INDEX — STRUCTURE → SAVE → LOAD')

# ── 1. Print structure ───────────────────────────────────────────────────────
print('\n━━ STRUCTURE ━━')
print(f'  type   : {type(dense_index)}')
print(f'  shape  : {dense_index.shape}   ← (n_chunks, embedding_dim)')
print(f'  dtype  : {dense_index.dtype}')
print(f'  RAM    : {dense_index.nbytes / 1024:.1f} KB')

np.set_printoptions(precision=6, suppress=True, linewidth=120)
for i, vec in enumerate(dense_index):
    print(f'\n  chunk {i}  ({len(vec)} dims)')
    print(f'    first 10 : {np.round(vec[:10], 6).tolist()}')
    print(f'    last  10 : {np.round(vec[-10:], 6).tolist()}')
    print(f'    min={vec.min():.6f}  max={vec.max():.6f}  '
          f'mean={vec.mean():.6f}  L2={np.linalg.norm(vec):.6f}')

print('\n  full matrix stored in .npy (every row = one chunk embedding):')
print(dense_index)

# ── 2. Save (.npy stores the raw float matrix, faster + smaller than pickle) ─
print('\n━━ SAVE ━━')
np.save('dense_index.npy', dense_index)
with open('all_chunks.pkl', 'wb') as f:
    pickle.dump(ALL_CHUNKS, f)
print('  saved → dense_index.npy   (float32 matrix, the vectors only)')
print('  saved → all_chunks.pkl    (chunk text + metadata, maps row index → text)')

# ── 3. Load and verify ───────────────────────────────────────────────────────
print('\n━━ LOAD ━━')
dense_loaded  = np.load('dense_index.npy')
with open('all_chunks.pkl', 'rb') as f:
    chunks_loaded = pickle.load(f)
print(f'  shape   : {dense_loaded.shape}')
print(f'  dtype   : {dense_loaded.dtype}')
print(f'  RAM     : {dense_loaded.nbytes / 1024:.1f} KB')
print(f'  chunks  : {len(chunks_loaded)} entries loaded')
assert np.allclose(dense_loaded, dense_index)
print('  ✅ loaded dense matrix matches original')


══════════════════════════════════════════════════════════════════════
  DENSE INDEX — STRUCTURE → SAVE → LOAD
══════════════════════════════════════════════════════════════════════

━━ STRUCTURE ━━
  type   : <class 'numpy.ndarray'>
  shape  : (24, 384)   ← (n_chunks, embedding_dim)
  dtype  : float32
  RAM    : 36.0 KB

  chunk 0  (384 dims)
    first 10 : [-0.024304000660777092, 0.00496299983933568, 0.01028400007635355, 0.003360999980941415, -0.027615999802947044, -0.049584001302719116, 0.09102100133895874, -0.09627500176429749, 0.04264099895954132, -0.026371000334620476]
    last  10 : [-0.08661799877882004, 0.03889400139451027, -0.044537998735904694, -0.06386599689722061, -0.05725499987602234, -0.0007249999907799065, 0.05481899902224541, -0.027580000460147858, 0.0549359992146492, 0.012903999537229538]
    min=-0.192223  max=0.168813  mean=-0.000649  L2=1.000000

  chunk 1  (384 dims)
    first 10 : [-0.05665700137615204, 0.04646899923682213, -0.008181000128388405, 0.0417950004339

## Stage 3c: Hybrid setup (OFFLINE, no extra storage needed)

In [15]:
banner('STAGE 3c — HYBRID INDEX  ← OFFLINE')
print()
print('  Hybrid retrieval = BM25 sparse + Dense cosine, linearly blended.')
print('  No additional index to build — the merge happens at query time.')
print()
print('  HYBRID_SCORE(q, chunk) = α × norm(BM25) + (1-α) × norm(Dense)')
print('  α ∈ [0,1]:  α=1 → pure BM25   |  α=0 → pure dense')
print()

ALPHA = 0.4
print(f'  α = {ALPHA}  →  HYBRID = {ALPHA} × BM25 + {1-ALPHA} × Dense')
print()
print('━'*70)
print('  ⏹  OFFLINE PHASE COMPLETE. Indexes saved. No rebuilding from here.')
print('━'*70)


══════════════════════════════════════════════════════════════════════
  STAGE 3c — HYBRID INDEX  ← OFFLINE
══════════════════════════════════════════════════════════════════════

  Hybrid retrieval = BM25 sparse + Dense cosine, linearly blended.
  No additional index to build — the merge happens at query time.

  HYBRID_SCORE(q, chunk) = α × norm(BM25) + (1-α) × norm(Dense)
  α ∈ [0,1]:  α=1 → pure BM25   |  α=0 → pure dense

  α = 0.4  →  HYBRID = 0.4 × BM25 + 0.6 × Dense

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  ⏹  OFFLINE PHASE COMPLETE. Indexes saved. No rebuilding from here.
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


## Stage 4: Retrieval functions

In [16]:
def retrieve_sparse(query, top_k=20):
    """
    BM25 sparse retrieval — keyword matching only.

    Scores every chunk using the BM25 formula (TF × IDF with length
    normalisation). Returns only chunks with score > 0, meaning at least
    one query token must appear in the chunk.
    """
    tokens = query.lower().split()          # must match how corpus was tokenised
    scores = bm25_index.get_scores(tokens)  # BM25 score for every chunk → float array
    ranked = np.argsort(scores)[::-1]       # sort descending by score
    ranked = ranked[:top_k]
    return [
        (ALL_CHUNKS[i], float(scores[i]))
        for i in ranked if scores[i] > 0    # score=0 means zero term overlap → skip
    ]


def retrieve_dense(query, top_k=20):
    """
    Dense semantic retrieval — embedding cosine similarity.

    Encodes the query with the same model used to build dense_index,
    then computes cosine similarity against every chunk vector.
    Always returns top_k results (no zero-score filter) because cosine
    similarity is defined for every pair.
    """
    query_vec = encoder.encode([query])                   # shape: (1, 384)
    sims      = cosine_similarity(query_vec, dense_index)[0]  # shape: (n_chunks,)
    ranked    = np.argsort(sims)[::-1][:top_k]
    return [(ALL_CHUNKS[i], float(sims[i])) for i in ranked]


def retrieve_hybrid(query, top_k=20, alpha=ALPHA):
    """
    Hybrid retrieval — weighted fusion of BM25 and dense scores.

    Both score arrays are normalised to [0,1] before fusion so they are
    on the same scale. BM25 uses max-normalisation (always ≥ 0); dense
    uses min-max normalisation (cosine can be negative).
    """
    # ── Sparse leg ──────────────────────────────────────────────────────
    tokens  = query.lower().split()
    bm25_s  = bm25_index.get_scores(tokens)
    bm25_s  = bm25_s / (bm25_s.max() + 1e-9)             # max-norm → [0, 1]

    # ── Dense leg ───────────────────────────────────────────────────────
    qvec    = encoder.encode([query])
    dense_s = cosine_similarity(qvec, dense_index)[0]
    dense_s = (dense_s - dense_s.min()) / \
              (dense_s.max() - dense_s.min() + 1e-9)      # min-max → [0, 1]

    # ── Fusion ──────────────────────────────────────────────────────────
    hybrid_s = alpha * bm25_s + (1 - alpha) * dense_s
    ranked   = np.argsort(hybrid_s)[::-1][:top_k]
    return [(ALL_CHUNKS[i], float(hybrid_s[i])) for i in ranked]


print('✅ Retrieval functions defined: retrieve_sparse | retrieve_dense | retrieve_hybrid')

✅ Retrieval functions defined: retrieve_sparse | retrieve_dense | retrieve_hybrid


## Stage 4 demo: run all three retrievers on a KFP query

In [17]:
banner('STAGE 4 — RETRIEVAL  ← ONLINE')

QUERY = "What is Tigress's fighting style and why does she resent Po?"
TOP_K = 5

print(f'\n  Query: "{QUERY}"')
print()

sparse_results = retrieve_sparse(QUERY, top_k=TOP_K)
dense_results  = retrieve_dense (QUERY, top_k=TOP_K)
hybrid_results = retrieve_hybrid(QUERY, top_k=TOP_K)


def show_results(label, results):
    """Pretty-print retrieval results with rank, score, source, and text preview."""
    print(f'  ── {label} (top {len(results)}) {"-"*(50-len(label))}')
    for rank, (chunk, score) in enumerate(results, 1):
        preview = chunk['text'][:85].replace('\n', ' ')
        print(f'    #{rank}  score={score:.3f}  [{chunk["id"]}]')
        print(f'        "{preview}..."')
    print()


show_results('SPARSE (BM25)',                       sparse_results)
show_results('DENSE  (Sentence-Transformer)',        dense_results)
show_results(f'HYBRID (α={ALPHA} BM25 + {1-ALPHA} Dense)', hybrid_results)

print('  📌 Watch whether noisy KF docs (doc_G/doc_H) surface in results.')
print('     They contain "kung fu" but not KFP-specific terms.')
print('     Dense should penalise them; sparse may partially surface them.')


══════════════════════════════════════════════════════════════════════
  STAGE 4 — RETRIEVAL  ← ONLINE
══════════════════════════════════════════════════════════════════════

  Query: "What is Tigress's fighting style and why does she resent Po?"

  ── SPARSE (BM25) (top 5) -------------------------------------
    #1  score=6.918  [doc_B_tigress_c2]
        "Shifu and to the Valley of Peace. Over time Tigress develops respect for Po, though s..."
    #2  score=5.451  [doc_B_tigress_c0]
        "Tigress is the unofficial leader of the Furious Five and the strongest fighter among ..."
    #3  score=4.130  [doc_B_tigress_c1]
        "style is direct and powerful, relying on strength, speed, and iron-hard discipline. T..."
    #4  score=3.759  [doc_C_mantis_c0]
        "Mantis is the smallest member of the Furious Five but arguably the fastest. His fight..."
    #5  score=3.657  [doc_E_furious_five_c1]
        "direct power, Crane relies on evasion and aerial moves, Mantis uses speed and

## Vocabulary mismatch demo + noisy document resilience

In [18]:
banner('RETRIEVAL — Vocabulary Mismatch + Noise Resilience Demo')

# Query uses paraphrase, not exact character names
mismatch_query = "the panda who was chosen as the chosen warrior"
print(f'  Query A (paraphrase): "{mismatch_query}"')
print()

sp_res = retrieve_sparse(mismatch_query, top_k=3)
de_res = retrieve_dense (mismatch_query, top_k=3)

print('  SPARSE — requires exact word overlap:')
if sp_res:
    for chunk, score in sp_res:
        print(f'    score={score:.3f}  [{chunk["source"]}]: "{chunk["text"][:75]}..."')
else:
    print('    ❌ No results — zero term overlap with any chunk')

print('\n  DENSE — semantic match:')
for chunk, score in de_res:
    print(f'    score={score:.3f}  [{chunk["source"]}]: "{chunk["text"][:75]}..."')

print()
# Query that should surface noisy docs under sparse but not dense
noise_query = "kung fu training monastery techniques"
print(f'  Query B (noise-prone): "{noise_query}"')
print()
sp_noise = retrieve_sparse(noise_query, top_k=5)
de_noise = retrieve_dense (noise_query, top_k=5)

print('  SPARSE results (may surface noisy Shaolin/WingChun docs):')
for chunk, score in sp_noise:
    tag = '🔴 NOISY' if chunk['source'] in ['doc_G_shaolin_history','doc_H_wingchun_techniques'] else '🐼 KFP  '
    print(f'    {tag}  score={score:.3f}  [{chunk["source"]}]')

print('\n  DENSE results (semantic context should push noisy docs down):')
for chunk, score in de_noise:
    tag = '🔴 NOISY' if chunk['source'] in ['doc_G_shaolin_history','doc_H_wingchun_techniques'] else '🐼 KFP  '
    print(f'    {tag}  score={score:.3f}  [{chunk["source"]}]')


══════════════════════════════════════════════════════════════════════
  RETRIEVAL — Vocabulary Mismatch + Noise Resilience Demo
══════════════════════════════════════════════════════════════════════
  Query A (paraphrase): "the panda who was chosen as the chosen warrior"

  SPARSE — requires exact word overlap:
    score=8.721  [doc_B_tigress]: "Tigress is the unofficial leader of the Furious Five and the strongest figh..."
    score=7.724  [doc_A_po]: "Po is a giant panda who works as a noodle waiter in his adoptive father Mr ..."
    score=6.868  [doc_F_shifu_oogway]: "master who ever lived. Oogway chose Po as the Dragon Warrior based on a vis..."

  DENSE — semantic match:
    score=0.579  [doc_D_tai_lung]: "Tai Lung is the primary antagonist of the first Kung Fu Panda film. He was ..."
    score=0.546  [doc_A_po]: "Po is a giant panda who works as a noodle waiter in his adoptive father Mr ..."
    score=0.520  [doc_A_po]: "Po learns that the Dragon Scroll contains no secret — the

## Stage 5: Cross-encoder reranking

In [19]:
banner('STAGE 5 — RERANKING  ← ONLINE (cross-encoder)')
print()
print('  Bi-encoder (retriever): encodes query and doc separately → fast, approximate.')
print('  Cross-encoder (reranker): encodes (query, doc) jointly → slow, highly accurate.')
print('  Pipeline: retrieve top-20 fast  →  rerank 20 precisely  →  keep top-3 for LM.')
print()

print('  Loading cross-encoder: cross-encoder/ms-marco-MiniLM-L-6-v2 ...')
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
print('  ✅ Cross-encoder loaded')
print()

QUERY = "How did Po defeat Tai Lung?"

candidate_results = retrieve_hybrid(QUERY, top_k=20)
candidate_chunks  = [r[0] for r in candidate_results]

print(f'  Query: "{QUERY}"')
print(f'  Step 1: hybrid retriever → {len(candidate_chunks)} candidates')
print(f'  Step 2: cross-encoder scores each (query, chunk) pair...')

pairs  = [(QUERY, c['text']) for c in candidate_chunks]
scores = reranker.predict(pairs)
ranked_indices = np.argsort(scores)[::-1]

section('Before reranking — hybrid retriever order (top 5)')
for rank, (chunk, ret_score) in enumerate(candidate_results[:5], 1):
    orig_idx   = candidate_chunks.index(chunk)
    rerank_scr = scores[orig_idx]
    print(f'  #{rank}  retriever={ret_score:.3f}  reranker={rerank_scr:.2f}  [{chunk["id"]}]')
    print(f'      "{chunk["text"][:80]}..."')

print()
section('After reranking — cross-encoder order (top 3)')
TOP_AFTER_RERANK = 3
final_chunks = []
for rank, idx in enumerate(ranked_indices[:TOP_AFTER_RERANK], 1):
    chunk = candidate_chunks[idx]
    final_chunks.append(chunk)
    print(f'  #{rank}  reranker={scores[idx]:.2f}  [{chunk["id"]}]')
    print(f'      "{chunk["text"][:100]}..."')
    print()

print(f'  Pruned: {len(candidate_chunks)} → {TOP_AFTER_RERANK} chunks for generation')


══════════════════════════════════════════════════════════════════════
  STAGE 5 — RERANKING  ← ONLINE (cross-encoder)
══════════════════════════════════════════════════════════════════════

  Bi-encoder (retriever): encodes query and doc separately → fast, approximate.
  Cross-encoder (reranker): encodes (query, doc) jointly → slow, highly accurate.
  Pipeline: retrieve top-20 fast  →  rerank 20 precisely  →  keep top-3 for LM.

  Loading cross-encoder: cross-encoder/ms-marco-MiniLM-L-6-v2 ...


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 12566.04it/s]


  ✅ Cross-encoder loaded

  Query: "How did Po defeat Tai Lung?"
  Step 1: hybrid retriever → 20 candidates
  Step 2: cross-encoder scores each (query, chunk) pair...

  ┌─ Before reranking — hybrid retriever order (top 5) -----------
  #1  retriever=0.874  reranker=6.78  [doc_A_po_c1]
      "martial arts training. Master Oogway chooses Po as the Dragon Warrior despite th..."
  #2  retriever=0.837  reranker=5.78  [doc_D_tai_lung_c1]
      "peace, Tai Lung flew into a rage and attacked the Valley of Peace. He was defeat..."
  #3  retriever=0.787  reranker=3.25  [doc_F_shifu_oogway_c0]
      "Master Shifu is a red panda and the headmaster of the Jade Palace. He trained Ta..."
  #4  retriever=0.760  reranker=6.18  [doc_D_tai_lung_c2]
      "deepest wound is not physical but emotional: he craved Shifu's approval and neve..."
  #5  retriever=0.752  reranker=5.10  [doc_E_furious_five_c2]
      "come to respect his courage and creativity. The Five are defeated by Tai Lung on..."


  ┌─ After 

## Reranking rank-change table (aesthetic, separate cell)

In [20]:
banner('RERANKING — Rank Change Visualisation')
print()
print('  Chunk                        Retriever   Reranker   Movement')
print('  ' + '─'*65)

retriever_order = {c['id']: i+1 for i, c in enumerate(candidate_chunks)}
reranker_order  = {candidate_chunks[idx]['id']: i+1
                   for i, idx in enumerate(ranked_indices)}

for cid in list(retriever_order.keys())[:10]:
    r_rank  = retriever_order[cid]
    rr_rank = reranker_order[cid]
    diff    = r_rank - rr_rank
    arrow   = f'▲ +{diff}' if diff > 0 else (f'▼ {diff}' if diff < 0 else '  =')
    print(f'  {cid:<30} #{r_rank:<4}       #{rr_rank:<4}      {arrow}')


══════════════════════════════════════════════════════════════════════
  RERANKING — Rank Change Visualisation
══════════════════════════════════════════════════════════════════════

  Chunk                        Retriever   Reranker   Movement
  ─────────────────────────────────────────────────────────────────
  doc_A_po_c1                    #1          #1           =
  doc_D_tai_lung_c1              #2          #3         ▼ -1
  doc_F_shifu_oogway_c0          #3          #6         ▼ -3
  doc_D_tai_lung_c2              #4          #2         ▲ +2
  doc_E_furious_five_c2          #5          #4         ▲ +1
  doc_D_tai_lung_c0              #6          #7         ▼ -1
  doc_A_po_c0                    #7          #10        ▼ -3
  doc_F_shifu_oogway_c1          #8          #8           =
  doc_A_po_c2                    #9          #9           =
  doc_F_shifu_oogway_c2          #10         #12        ▼ -2


## Stage 6: Generation

In [ ]:
from transformers import pipeline as hf_pipeline

banner('STAGE 6 — GENERATION  ← ONLINE')
print()
print('  Loading Qwen2-1.5B-Instruct...')
generator = hf_pipeline(
    'text-generation',
    model     = 'Qwen/Qwen2-1.5B-Instruct',
    device_map= 'auto',
    max_new_tokens=150,
    do_sample = False,
)
print('  ✅ Generator loaded')


def build_prompt(query, chunks):
    """
    Build a RAG prompt: system instruction + numbered context blocks + question.
    The LM is instructed to answer ONLY from the provided context to prevent
    hallucination from its training data.
    """
    context_block = '\n'.join(
        [f'  [{i+1}] {c["text"]}' for i, c in enumerate(chunks)]
    )
    return (
        f'Answer the question using ONLY the provided context. Be concise.\n\n'
        f'Context:\n{context_block}\n\n'
        f'Question: {query}\n\nAnswer:'
    )


QUERY  = "How did Po defeat Tai Lung?"
prompt = build_prompt(QUERY, final_chunks)

section('Prompt sent to LM')
for line in prompt.split('\n'):
    print(f'  {line}')

print('\n' + '─'*70 + '\n  Generating...\n' + '─'*70)
output = generator(prompt, return_full_text=False)
answer = output[0]['generated_text'].split('\n')[0].strip()

print(f'\n  ❓ Query : {QUERY}')
print(f'  💬 Answer: {answer}')


══════════════════════════════════════════════════════════════════════
  STAGE 6 — GENERATION  ← ONLINE
══════════════════════════════════════════════════════════════════════

  Loading Qwen2-1.5B-Instruct...


/Users/andrevarilla/Git Repos/natural-language-processsing/.venv/lib/python3.13/site-packages/huggingface_hub/file_download.py:731: UserWarning: Not enough free disk space to download the file. The expected file size is: 3087.47 MB. The target location /Users/andrevarilla/.cache/huggingface/hub/models--Qwen--Qwen2-1.5B-Instruct/blobs only has 1053.87 MB free disk space.
  warnings.warn(
/Users/andrevarilla/Git Repos/natural-language-processsing/.venv/lib/python3.13/site-packages/huggingface_hub/file_download.py:731: UserWarning: Not enough free disk space to download the file. The expected file size is: 3087.47 MB. The target location /Users/andrevarilla/.cache/huggingface/hub/models--Qwen--Qwen2-1.5B-Instruct/blobs only has 265.14 MB free disk space.
  warnings.warn(


## Full configurable RAG pipeline  Options exposed:   retriever_type   : 'sparse' | 'dense' | 'hybrid' | list for ensemble   chunk_size       : words per chunk (re-chunks corpus on demand)   overlap          : overlap words (re-chunks corpus on demand)   encoder_model    : any SentenceTransformer model name   top_retrieve     : candidates returned by retriever   top_final        : kept after reranking   verbose          : print stage-by-stage trace

In [ ]:
def build_index(chunk_size=60, overlap=15, encoder_model='all-MiniLM-L6-v2'):
    """
    (Re-)chunk the corpus and build fresh BM25 and dense indexes.

    Called automatically by rag_pipeline when non-default chunking params
    are requested. Allows exploration of how chunk_size and overlap affect
    retrieval quality without manually rebuilding indexes.
    """
    chunks = []
    for doc_id, text in raw_documents.items():
        clean = ' '.join(text.split())
        for i, c in enumerate(chunk_text(clean, chunk_size=chunk_size, overlap=overlap)):
            chunks.append({'id': f'{doc_id}_c{i}', 'text': c, 'source': doc_id})

    tokenised = [c['text'].lower().split() for c in chunks]
    bm25      = BM25Okapi(tokenised)

    enc   = SentenceTransformer(encoder_model)
    dense = enc.encode([c['text'] for c in chunks], show_progress_bar=False)

    return chunks, bm25, dense, enc


def rag_pipeline(
    query,
    retriever_type  = 'hybrid',     # 'sparse' | 'dense' | 'hybrid' | ['sparse','dense']
    chunk_size      = 60,           # words per chunk
    overlap         = 15,           # overlap words between chunks
    encoder_model   = 'all-MiniLM-L6-v2',
    alpha           = ALPHA,        # BM25 weight in hybrid (1-alpha = dense weight)
    top_retrieve    = 20,           # candidates from retriever
    top_final       = 3,            # kept after reranking
    verbose         = True,
):
    """
    Complete online RAG pipeline: retrieve → rerank → generate.

    Stages 1-3 (extraction, chunking, indexing) run offline.
    This function covers stages 4-6 only.

    If chunk_size or overlap differ from the global defaults, the corpus
    is re-chunked and re-indexed transparently so you can compare
    chunking strategies end-to-end.
    """
    # ── Rebuild indexes if chunking params differ from globals ───────────────
    use_global = (chunk_size == 60 and overlap == 15
                  and encoder_model == 'all-MiniLM-L6-v2')
    if use_global:
        chunks, bm25, dense, enc = ALL_CHUNKS, bm25_index, dense_index, encoder
    else:
        if verbose:
            print(f'  ⚙️  Rebuilding index: chunk_size={chunk_size}, overlap={overlap}, '
                  f'model={encoder_model}')
        chunks, bm25, dense, enc = build_index(chunk_size, overlap, encoder_model)

    if verbose:
        banner(f'RAG PIPELINE  |  retriever={retriever_type}  '
               f'chunk={chunk_size}/ol={overlap}')
        print(f'\n  Query: "{query}"')
        print(f'  Corpus: {len(chunks)} chunks  |  α={alpha}  |  '
              f'top_retrieve={top_retrieve}  top_final={top_final}')

    # ── Stage 4: Retrieve ────────────────────────────────────────────────────
    def _sparse(q, k):
        toks  = q.lower().split()
        sc    = bm25.get_scores(toks)
        idx   = np.argsort(sc)[::-1][:k]
        return [(chunks[i], float(sc[i])) for i in idx if sc[i] > 0]

    def _dense(q, k):
        qv  = enc.encode([q])
        sim = cosine_similarity(qv, dense)[0]
        idx = np.argsort(sim)[::-1][:k]
        return [(chunks[i], float(sim[i])) for i in idx]

    def _hybrid(q, k, a):
        toks = q.lower().split()
        bs   = bm25.get_scores(toks);  bs = bs / (bs.max() + 1e-9)
        qv   = enc.encode([q])
        ds   = cosine_similarity(qv, dense)[0]
        ds   = (ds - ds.min()) / (ds.max() - ds.min() + 1e-9)
        hs   = a * bs + (1 - a) * ds
        idx  = np.argsort(hs)[::-1][:k]
        return [(chunks[i], float(hs[i])) for i in idx]

    # Support single string or list of retrievers
    rtypes = [retriever_type] if isinstance(retriever_type, str) else retriever_type

    # Collect candidates across all requested retrievers, deduplicate by chunk id
    seen, candidates = set(), []
    for rt in rtypes:
        if rt == 'sparse':   res = _sparse(query, top_retrieve)
        elif rt == 'dense':  res = _dense (query, top_retrieve)
        else:                res = _hybrid(query, top_retrieve, alpha)
        for chunk, score in res:
            if chunk['id'] not in seen:
                seen.add(chunk['id'])
                candidates.append(chunk)

    if verbose:
        print(f'\n  Stage 4 — {len(candidates)} unique candidates from {rtypes}')

    # ── Stage 5: Rerank ──────────────────────────────────────────────────────
    pairs      = [(query, c['text']) for c in candidates]
    rr_scores  = reranker.predict(pairs)
    top_idx    = np.argsort(rr_scores)[::-1][:top_final]
    final      = [candidates[i] for i in top_idx]

    if verbose:
        print(f'  Stage 5 — Reranked → kept top {top_final}')
        for rank, chunk in enumerate(final, 1):
            noise = '🔴' if chunk['source'] in ['doc_G_shaolin_history',
                                                  'doc_H_wingchun_techniques'] else '🐼'
            print(f'    #{rank} {noise} [{chunk["id"]}]: "{chunk["text"][:70]}..."')

    # ── Stage 6: Generate ────────────────────────────────────────────────────
    prompt = build_prompt(query, final)
    out    = generator(prompt, return_full_text=False)
    answer = out[0]['generated_text'].split('\n')[0].strip()

    if verbose:
        print(f'\n  Stage 6 — Answer:')
        print(f'  💬 {answer}')

    return answer

## Run KFP queries through the pipeline

In [ ]:
banner('KFP QUERIES — Full Pipeline')

kfp_queries = [
    "What is the main weakness of BM25 retrieval?",         # should miss KFP docs
    "Who trained the Furious Five at the Jade Palace?",
    "How does Mantis use acupuncture to help Po?",
    "Why did Tai Lung attack the Valley of Peace?",
    "What did Tigress feel when Po was chosen as Dragon Warrior?",
    "How did Po defeat Tai Lung in the final battle?",
]

for q in kfp_queries:
    ans = rag_pipeline(q, retriever_type='hybrid', verbose=False)
    print(f'  ❓ {q}')
    print(f'  💬 {ans}')
    print()

## Chunking overlap comparison: same query, different overlap

In [ ]:
banner('CHUNKING OVERLAP COMPARISON — same query, three overlap settings')

overlap_query = "Tigress's fighting style relies on strength and iron discipline"
print(f'  Query: "{overlap_query}"\n')
print(f'  {"Overlap":>8}  {"Chunks":>7}  Answer')
print('  ' + '─'*70)

for ol in [0, 15, 30]:
    ans = rag_pipeline(
        overlap_query,
        retriever_type='hybrid',
        chunk_size=60,
        overlap=ol,
        verbose=False,
    )
    n_chunks = sum(
        len(chunk_text(' '.join(t.split()), chunk_size=60, overlap=ol))
        for t in raw_documents.values()
    )
    print(f'  overlap={ol:>2}   {n_chunks:>5} chunks   {ans[:80]}')

## Multi-retriever ensemble + encoder comparison

In [ ]:
banner('RETRIEVER ENSEMBLE — sparse + dense combined')

ensemble_query = "What secret did Oogway reveal about the Dragon Scroll?"
print(f'  Query: "{ensemble_query}"\n')

for rtype in ['sparse', 'dense', 'hybrid', ['sparse', 'dense']]:
    label = '+'.join(rtype) if isinstance(rtype, list) else rtype
    ans   = rag_pipeline(ensemble_query, retriever_type=rtype, verbose=False)
    print(f'  [{label:>16}]  {ans[:85]}')

## Full comparison: all three retrievers on all KFP queries

In [ ]:
banner('FULL COMPARISON — Sparse vs Dense vs Hybrid on all KFP queries')

comparison_queries = [
    "Who is Tigress and what drives her?",
    "What technique did Po use to beat Tai Lung?",
    "How does Mantis fight despite his small size?",
    "What is the role of Shifu in training Po?",
    "Why are the noisy kung fu documents irrelevant to this question?",
]

print(f'  {"Query":<50}  {"sparse":>8}  {"dense":>8}  {"hybrid":>8}')
print('  ' + '─'*80)

for q in comparison_queries:
    short = q[:48]
    for rtype, col in [('sparse', None), ('dense', None), ('hybrid', None)]:
        pass  # just run and print inline below

    ans_sp = rag_pipeline(q, retriever_type='sparse', verbose=False)
    ans_de = rag_pipeline(q, retriever_type='dense',  verbose=False)
    ans_hy = rag_pipeline(q, retriever_type='hybrid', verbose=False)
    print(f'\n  ❓ {q}')
    print(f'     sparse : {ans_sp[:100]}')
    print(f'     dense  : {ans_de[:100]}')
    print(f'     hybrid : {ans_hy[:100]}')

## Pipeline summary (markdown-style)

In [ ]:
print("""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  PIPELINE RECAP
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  OFFLINE (run once when corpus changes):
    Stage 1  Extract text per document (PDF / HTML / DOCX → plain string)
    Stage 2  Chunk into overlapping windows
               chunk_size too small → context lost
               chunk_size too large → too much noise, LM confused
               overlap=0 → answers at boundaries split across chunks
               overlap>0 → boundary answers always retrievable, +storage
    Stage 3a BM25 sparse index   — no model, no GPU, dict + list in RAM
    Stage 3b Dense embedding matrix — GPU, 384-dim float32, saved to .npy
    Stage 3c Hybrid — no extra storage, merge at query time with alpha

  ONLINE (per query, milliseconds–seconds):
    Stage 4  Retrieve top-k from pre-built index (READ only, never rebuild)
    Stage 5  Cross-encoder reranks top-k pairs jointly → keep top-3
    Stage 6  LM reads [ctx1, ctx2, ctx3] + question → answer

  NOISE RESILIENCE:
    doc_G (Shaolin history) and doc_H (Wing Chun) are real-world kung fu
    documents irrelevant to KFP plot. Sparse retrieval surfaces them on
    generic 'kung fu' queries; dense retrieval suppresses them because
    KFP plot embeddings cluster away from real-world martial arts text.
    Hybrid with low alpha (α < 0.5) inherits dense's noise resistance.

  COST LADDER:
    BM25   ($)   — instant, no model, exact match only
    Dense  ($$)  — encoder model, semantic match, needs training data
    Reranker ($$$) — cross-encoder, most accurate, runs per query pair
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
""")